# Klasifikasi Hasil Pertandingan Top Lima Liga Eropa Musim 2025/2026 Berdasarkan Statistik Pertandingan Menggunakan Random Forest dengan Optimasi Hyperparameter Bayesian

**Topik Penelitian:** Klasifikasi Prediksi Hasil Pertandingan Sepak Bola Top 5 Liga Eropa
**Dataset:** `data top 5 liga.csv` (1.752 sampel, 22 atribut)
**Tujuan:** Membandingkan performa model klasifikasi Random Forest baseline dan Random Forest teroptimasi Bayesian (Optuna).

## 1. Import Library & Persiapan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Memuat & Pra-pemrosesan Data Top 5 Liga

In [ ]:
df = pd.read_csv('../data top 5 liga.csv', sep=';')
df = df[df['Date'] != 'Date'].dropna(subset=['FTR']).copy()

feature_cols = ['HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=feature_cols)

X = df[feature_cols]
le = LabelEncoder()
y = le.fit_transform(df['FTR'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

print(f"Ukuran X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")

## 3. Pemodelan Random Forest & Optimasi Bayesian (Optuna)

In [ ]:
# Baseline Random Forest
rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train_scaled, y_train)

# Bayesian Optimization dengan Optuna
def objective(trial):
    clf = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 20, 300),
        max_depth=trial.suggest_int('max_depth', 2, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 15),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        criterion=trial.suggest_categorical('criterion', ['gini', 'entropy']),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        random_state=42
    )
    return cross_val_score(clf, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

rf_bayes = RandomForestClassifier(**study.best_params, random_state=42)
rf_bayes.fit(X_train_scaled, y_train)

print("Best Hyperparameters (Bayesian Optimization):", study.best_params)

## 4. Evaluasi & Visualisasi Feature Importance

In [ ]:
y_pred_base = rf_base.predict(X_test_scaled)
y_pred_bayes = rf_bayes.predict(X_test_scaled)

acc_base = accuracy_score(y_test, y_pred_base)
acc_bayes = accuracy_score(y_test, y_pred_bayes)

print(f"Akurasi Random Forest (Baseline): {acc_base:.4f}")
print(f"Akurasi Random Forest (Bayesian Opt): {acc_bayes:.4f}")